# Run MAGICC
In this notebook, we will run MAGICC for a single scenario for a small number of ensemble members for MAGICC7.

In [1]:
ENSEMBLE_MEMBERS = 600
MODEL = "AIM/CGE V2.2"
SCENARIO = "ENGAGE-Feasibility-Maximum-Effort/Institutional_NZCO2"
FILE = "output/102_netzero_co2_extended.csv"
CO2_ONLY = False

In [2]:
import multiprocessing
import os
import platform
from functools import partial
from pathlib import Path

import openscm_units
import pandas_indexing as pix
import pandas_openscm
import pint
import seaborn as sns
from pandas_openscm.io import load_timeseries_csv
import pandas as pd

import gcages.cmip7_scenariomip.pre_processing.reaggregation.basic
from gcages.cmip7_scenariomip.harmonisation import (
    create_cmip7_scenariomip_global_harmoniser,
)
from gcages.cmip7_scenariomip.infilling import CMIP7ScenarioMIPInfiller
from gcages.cmip7_scenariomip.post_processing import CMIP7ScenarioMIPPostProcessor
from gcages.cmip7_scenariomip.pre_processing import CMIP7ScenarioMIPPreProcessor
from gcages.cmip7_scenariomip.scm_running import CMIP7ScenarioMIPSCMRunner
from gcages.index_manipulation import split_sectors

In [3]:
BASE_DIR = Path("../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs")
CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS = BASE_DIR / "history_cmip7_scenariomip.csv"

In [4]:
start = load_timeseries_csv(
    FILE,
    index_columns=["model", "scenario", "region", "variable", "unit"],
    out_columns_type=int,
    out_columns_name="year"
)

In [5]:
scenario_to_run = (
    start
    .loc[
        pd.IndexSlice[MODEL, SCENARIO, :, :, :], :
    ]
)

In [6]:
scenario_to_run

year                                                                                                                    2023  \
model        scenario                                           region variable                unit                            
AIM/CGE V2.2 ENGAGE-Feasibility-Maximum-Effort/Institutional... World  Emissions|BC            Mt BC/yr             7.219245   
                                                                       Emissions|C2F6          kt C2F6/yr           2.431873   
                                                                       Emissions|C3F8          kt C3F8/yr           0.721178   
                                                                       Emissions|C4F10         kt C4F10/yr          0.101586   
                                                                       Emissions|C5F12         kt C5F12/yr          0.061383   
                                                                       Emissions|C6F14         kt C6F14/yr          0.062998   
                                                                       Emissions|C7F16         kt C7F16/yr          0.186453   
                                                                       Emissions|C8F18         kt C8F18/yr          0.136638   
                                                                       Emissions|CCl4          kt CCl4/yr          40.297308   
                                                                       Emissions|CF4           kt CF4/yr           15.117718   
                                                                       Emissions|CFC11         kt CFC11/yr         55.993337   
                                                                       Emissions|CFC113        kt CFC113/yr         2.030656   
                                                                       Emissions|CFC114        kt CFC114/yr         0.873953   
                                                                       Emissions|CFC115        kt CFC115/yr         0.027629   
                                                                       Emissions|CFC12         kt CFC12/yr         20.278901   
                                                                       Emissions|CH2Cl2        kt CH2Cl2/yr      1612.125558   
                                                                       Emissions|CH3Br         kt CH3Br/yr        140.486974   
                                                                       Emissions|CH3CCl3       kt CH3CCl3/yr        1.925368   
                                                                       Emissions|CH3Cl         kt CH3Cl/yr       5376.602241   
                                                                       Emissions|CH4           Mt CH4/yr          363.489901   
                                                                       Emissions|CHCl3         kt CHCl3/yr        400.499380   
                                                                       Emissions|CO            Mt CO/yr           774.053434   
                                                                       Emissions|CO2|Biosphere Mt CO2/yr         3627.550667   
                                                                       Emissions|CO2|Fossil    Mt CO2/yr        38712.217040   
                                                                       Emissions|HCFC141b      kt HCFC141b/yr      57.429166   
                                                                       Emissions|HCFC142b      kt HCFC142b/yr      20.176795   
                                                                       Emissions|HCFC22        kt HCFC22/yr       324.526866   
                                                                       Emissions|HFC125        kt HFC125/yr       124.074300   
                                                                       Emissions|HFC134a       kt HFC134a/yr      291.788600   
                                

In [7]:
MAGICC_EXE_PATH = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/magicc-v7.6.0a3/bin"
)
MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/magicc-v7.6.0a3/configs/magicc-ar7-fast-track-drawnset-v0-3-0.json"
)

In [8]:
if platform.system() == "Darwin":
    if platform.processor() == "arm":
        MAGICC_EXE = MAGICC_EXE_PATH / "magicc-darwin-arm64"
        os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/Cellar/gcc/15.2.0_1/lib/gcc/current"

elif platform.system() == "Linux":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc"

elif platform.system() == "Windows":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc.exe"

In [9]:
scm_runner = CMIP7ScenarioMIPSCMRunner.from_cmip7_scenariomip_config(
    # Generally, you want to run SCMs in parallel
    n_processes=multiprocessing.cpu_count(),
    magicc_exe_path=MAGICC_EXE,
    magicc_prob_distribution_path=MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE,
    historical_emissions_path=CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS,
    output_variables=("Surface Air Temperature Change",),
    batch_size_scenarios=15,
)

/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/scmdata/database/_database.py:9: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  import tqdm.autonotebook as tqdman
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [10]:
scm_runner.climate_models_cfgs["MAGICC7"] = scm_runner.climate_models_cfgs[
        "MAGICC7"
][:ENSEMBLE_MEMBERS]

In [11]:
scm_results = scm_runner(scenario_to_run)
scm_results

Climate models:   0%|          | 0/1 [00:00<?, ?it/s]
[WARNING] 10:33:36 - openscm_runner.adapters.magicc7.magicc7: Historical data has not been checked
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/scmdata/run.py:2632: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([ret._meta.to_frame(), *to_join_metas]).astype("category")


Writing SCEN7 files: 100%|██████████| 1.00/1.00 [00:00<00:00, 6.04it/s]


/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
Front serial:

time                                                                                                                              1750  \
climate_model  model        region run_id scenario                                           unit variable                               
MAGICCv7.6.0a3 AIM/CGE V2.2 World  0      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   1      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   2      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   3      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   4      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
...                                                                                                                                ...   
                                   595    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   596    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   597    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   598    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   
                                   599    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change   0.0   

time                                                                                                                                  1751  \
climate_model  model        region run_id scenario                                           unit variable                                   
MAGICCv7.6.0a3 AIM/CGE V2.2 World  0      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.025523   
                                   1      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.031040   
                                   2      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.030692   
                                   3      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.031590   
                                   4      ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.027147   
...                                                                                                                                    ...   
                                   595    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.017827   
                                   596    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.026252   
                                   597    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.030374   
                                   598    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.028743   
                                   599    ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Air Temperature Change  0.022891   

time                                                                                                                                  1752  \
climate_model  model        region run_id scenario                                           unit variable                                   
MAGICCv7.6.0a3 AIM/CGE V2.2 World  0      ENGAGE-Feasibility-Maximum-Effor

In [12]:
post_processor = CMIP7ScenarioMIPPostProcessor.from_cmip7_scenariomip_config()
post_processed_results = post_processor(
    scm_results.loc[pix.isin(variable=["Surface Air Temperature Change"])]
)

In [13]:
post_processed_results.metadata_categories

climate_model   model         scenario                                               metric       
MAGICCv7.6.0a3  AIM/CGE V2.2  ENGAGE-Feasibility-Maximum-Effort/Institutional_NZCO2  category_name    C3: limit warming to 2°C (>67%)
                                                                                     category                                      C3
dtype: object

In [14]:
post_processed_results.metadata_quantile.loc[
    pix.isin(quantile=[0.05, 0.5, 0.95])
].unstack(["quantile", "metric"]).round(2).sort_index(axis="columns")

quantile                                                                                                               0.05  \
metric                                                                                                                 2100   
climate_model  model        region scenario                                           unit variable                           
MAGICCv7.6.0a3 AIM/CGE V2.2 World  ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Temperature (GSAT)  1.11   
                                                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                                                     \
metric                                                                                                                  max   
climate_model  model        region scenario                                           unit variable                           
MAGICCv7.6.0a3 AIM/CGE V2.2 World  ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Temperature (GSAT)  1.28   
                                                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                                                        \
metric                                                                                                                max_year   
climate_model  model        region scenario                                           unit variable                              
MAGICCv7.6.0a3 AIM/CGE V2.2 World  ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Temperature (GSAT)      NaN   
                                                                                      yr   Surface Temperature (GSAT)   2047.0   

quantile                                                                                                               0.50  \
metric                                                                                                                 2100   
climate_model  model        region scenario                                           unit variable                           
MAGICCv7.6.0a3 AIM/CGE V2.2 World  ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Temperature (GSAT)  1.58   
                                                                                      yr   Surface Temperature (GSAT)   NaN   

quantile                                                                                                                    \
metric                                                                                                                 max   
climate_model  model        region scenario                                           unit variable                          
MAGICCv7.6.0a3 AIM/CGE V2.2 World  ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Temperature (GSAT)  1.7   
                                                                                      yr   Surface Temperature (GSAT)  NaN   

quantile                                                                                                                        \
metric                                                                                                                max_year   
climate_model  model        region scenario                                           unit variable                              
MAGICCv7.6.0a3 AIM/CGE V2.2 World  ENGAGE-Feasibility-Maximum-Effort/Institutional... K    Surface Temperature (GSAT)      NaN   
                                                                                      yr   Surface Temperature (GSAT)   2049.0   

quantile                                                                                                               0.95  \
metric                          